# CAP6411 Workstream 1 — Reproducible BraTS DINOv3 Baseline

Sanitized team handoff notebook derived from the completed Colab workflow. It contains no dataset, checkpoint, credential, account identifier, or saved execution output. Expensive cells are guarded and are **not** executed automatically.

Completed full-cohort validation results (not test results):

| Metric | Epoch 1 | Epoch 2 |
|---|---:|---:|
| WT Dice | 0.84847853 | 0.86898142 |
| TC Dice | 0.81595193 | 0.85647742 |
| ET Dice | 0.71793982 | 0.73206816 |
| Macro Dice | 0.79412342 | 0.81917567 |

The isolated 188-patient test split has not been evaluated.


## 1. Setup

Clone or update the shared repository without detaching at an obsolete commit. Override `CAP6411_REPOSITORY_REF` if testing another reviewed branch.


In [ ]:
import os
import subprocess
from pathlib import Path

REPOSITORY_URL = "https://github.com/FaramarzAboutalebi/CAP6411-Adaptive-Token-Merging.git"
REPOSITORY_REF = os.environ.get("CAP6411_REPOSITORY_REF", "main")
REPO_ROOT = Path(os.environ.get("CAP6411_REPO_ROOT", "/content/CAP6411-Adaptive-Token-Merging"))

if not (REPO_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_REF, REPOSITORY_URL, str(REPO_ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPOSITORY_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPOSITORY_REF], check=True)

os.chdir(REPO_ROOT)
print("Repository:", REPO_ROOT)
print("Revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


### Install project dependencies

Use a platform-appropriate PyTorch build. The official DINOv3 checkout may require additional packages listed by its pinned source revision.


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)


### Obtain the official DINOv3 source

The code is public, but the required pretrained weights are gated. Request authorized access through Meta's official DINOv3 download process. Never place access URLs or tokens in this notebook.


In [ ]:
DINO_ROOT = Path(os.environ.get("DINOV3_ROOT", "/content/dinov3"))
DINO_REVISION = "6876159a11b4df116f30f667f8c9888617df0751"

if not (DINO_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "https://github.com/facebookresearch/dinov3.git", str(DINO_ROOT)],
        check=True,
    )
subprocess.run(["git", "-C", str(DINO_ROOT), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(DINO_ROOT), "checkout", DINO_REVISION], check=True)
assert subprocess.check_output(
    ["git", "-C", str(DINO_ROOT), "rev-parse", "HEAD"], text=True
).strip() == DINO_REVISION


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(DINO_ROOT), "--no-deps"], check=True)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "ftfy", "omegaconf", "regex", "scikit-learn",
        "submitit", "termcolor", "torchmetrics",
    ],
    check=True,
)


### Configure private assets

Keep these assets outside Git. Set `CAP6411_PRIVATE_ROOT` to a local directory or an authorized mounted private volume. The placeholders below are intentionally unusable until configured.


In [ ]:
PRIVATE_ROOT = Path(os.environ.get("CAP6411_PRIVATE_ROOT", "/path/to/private/CAP6411"))
BRATS_ZIP = PRIVATE_ROOT / "datasets/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData.zip"
SPLIT_MANIFEST = PRIVATE_ROOT / "brats2023_gli_split_seed42.json"
PRETRAINED_CHECKPOINT = PRIVATE_ROOT / "checkpoints/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"
TRAINED_CHECKPOINT = PRIVATE_ROOT / "checkpoints/full_cohort/dinov3_vitb16_brats_full_cohort/best.pt"
RUN_PRIVATE_DATA_CELLS = False
RUN_TRAINING = False

print("Set CAP6411_PRIVATE_ROOT before enabling private-data cells.")


### Validate the public repository

This test suite uses synthetic NIfTI data and a synthetic DINOv3-compatible fixture. It does not require gated weights or BraTS.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=REPO_ROOT,
    check=True,
)


## 2. Dataset and preprocessing

`BraTSAxialSliceDataset` reads one patient from the ZIP at a time and emits axial samples with images `[4,240,240]` in T1n, T1c, T2w, T2f order and overlapping targets `[3,240,240]` in WT, TC, ET order. Per-modality normalization is finite, nonzero-foreground z-scoring with zero background. Full-cohort work disables persistent disk caching.


In [ ]:
from cap6411_atm.data.dataset import BraTSAxialSliceDataset

if RUN_PRIVATE_DATA_CELLS:
    for required in (BRATS_ZIP, SPLIT_MANIFEST):
        assert required.is_file(), f"Missing authorized private input: {required}"
    validation_dataset = BraTSAxialSliceDataset(
        archive_path=BRATS_ZIP,
        manifest_path=SPLIT_MANIFEST,
        split="validation",
        inspection_config=REPO_ROOT / "configs/brats2023_gli_inspection.json",
        cache_dir=None,
        max_cached_patients=1,
    )
    sample = validation_dataset[0]
    assert tuple(sample["image"].shape) == (4, 240, 240)
    assert tuple(sample["target"].shape) == (3, 240, 240)
    print(sample["patient_id"], sample["slice_index"])
else:
    print("Skipped private BraTS loading. Set RUN_PRIVATE_DATA_CELLS=True after configuring paths.")


## 3. Model and stable feature interface

The official frozen DINOv3 ViT-B/16 encoder receives a trainable 4→3 MRI adapter. `extract_features` returns four normalized dense maps from blocks 2, 5, 8, and 11. `decode_features` consumes four `[B,768,15,15]` maps and returns raw WT/TC/ET logits `[B,3,240,240]`.


In [ ]:
import torch
from cap6411_atm.models import DINOv3SegmentationConfig, build_pretrained_brats_dinov3

model_config = DINOv3SegmentationConfig.from_json(
    REPO_ROOT / "configs/dinov3_vitb16_brats.json"
)

if RUN_PRIVATE_DATA_CELLS:
    assert PRETRAINED_CHECKPOINT.is_file(), "Obtain the authorized gated checkpoint first."
    model = build_pretrained_brats_dinov3(
        repository=DINO_ROOT,
        checkpoint=PRETRAINED_CHECKPOINT,
        config=model_config,
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    images = torch.zeros(1, 4, 240, 240, device=device)
    with torch.inference_mode():
        features = model.extract_features(images)
        logits = model.decode_features(features)
    assert all(tuple(feature.shape) == (1, 768, 15, 15) for feature in features)
    assert tuple(logits.shape) == (1, 3, 240, 240)
else:
    print("Skipped gated-weight model loading.")


### Develop against the interface without private weights

The synthetic fixture exercises class/storage-token handling, dense feature shapes, decoding, gradients, and checkpoint errors without downloading weights.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "unittest", "tests.test_dinov3_segmentation", "-v"],
    cwd=REPO_ROOT,
    check=True,
)


## 4. Materialize a private runtime configuration

Committed JSON files are templates. Create a runtime copy outside the repository so machine-specific paths and experiment destinations are never committed.


In [ ]:
import json
import tempfile


def runtime_config(template_name: str, *, experiment_name: str, epochs: int) -> Path:
    values = json.loads((REPO_ROOT / "configs" / template_name).read_text())
    values.update(
        archive_path=str(BRATS_ZIP),
        manifest_path=str(SPLIT_MANIFEST),
        inspection_config=str(REPO_ROOT / "configs/brats2023_gli_inspection.json"),
        model_config=str(REPO_ROOT / "configs/dinov3_vitb16_brats.json"),
        dinov3_repository=str(DINO_ROOT),
        checkpoint_path=str(PRETRAINED_CHECKPOINT),
        output_dir=str(PRIVATE_ROOT / "checkpoints"),
        experiment_name=experiment_name,
        epochs=epochs,
    )
    destination = Path(tempfile.gettempdir()) / f"{experiment_name}.json"
    destination.write_text(json.dumps(values, indent=2) + "\n")
    return destination

pilot_config = runtime_config(
    "brats2023_gli_baseline_training.json",
    experiment_name="dinov3_vitb16_brats_pilot",
    epochs=30,
)
full_config = runtime_config(
    "brats2023_gli_full_cohort_training.json",
    experiment_name="dinov3_vitb16_brats_full_cohort",
    epochs=1,
)
assert json.loads(full_config.read_text())["cache_dir"] is None
print("Runtime configs:", pilot_config, full_config)


## 5. One-batch smoke test and pilots

The smoke test performs one optimizer update. Run it in a fresh process, then start actual training in another fresh process. The committed pilot defaults select 8 train and 4 validation patients deterministically; an expanded pilot uses 32 and 8 through a private runtime config override.


In [ ]:
if RUN_TRAINING:
    subprocess.run(
        [sys.executable, "-m", "cap6411_atm.training.baseline", "--config", str(pilot_config), "--smoke-test"],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print("Training smoke test skipped. Set RUN_TRAINING=True intentionally.")


In [ ]:
if RUN_TRAINING:
    expanded = json.loads(pilot_config.read_text())
    expanded.update(
        experiment_name="dinov3_vitb16_brats_pilot32",
        pilot_train_patients=32,
        pilot_validation_patients=8,
    )
    expanded_path = Path(tempfile.gettempdir()) / "dinov3_vitb16_brats_pilot32.json"
    expanded_path.write_text(json.dumps(expanded, indent=2) + "\n")
    subprocess.run(
        [sys.executable, "-m", "cap6411_atm.training.baseline", "--config", str(expanded_path)],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print("Expanded pilot skipped.")


## 6. Full-cohort training

The storage-safe template uses all 876 train and 187 validation patients, batch size 2, CUDA AMP, and `cache_dir: null`. It preserves the one-patient in-memory cache but does not create persistent normalized `.npy` files.


In [ ]:
if RUN_TRAINING:
    subprocess.run(
        [sys.executable, "-m", "cap6411_atm.training.baseline", "--config", str(full_config)],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print("Full-cohort training skipped.")


### Resume without repeating completed epochs

The checkpoint stores a zero-based completed epoch index. `resume_from_checkpoint` returns `checkpoint["epoch"] + 1`; the RNG restoration fix moves CPU and CUDA RNG states to CPU tensors before restoring them. The runtime config's `epochs` is the desired total number of epochs.


In [ ]:
if RUN_TRAINING:
    resume_checkpoint = PRIVATE_ROOT / "checkpoints/dinov3_vitb16_brats_full_cohort/last.pt"
    assert resume_checkpoint.is_file()
    try:
        saved = torch.load(resume_checkpoint, map_location="cpu", weights_only=False)
    except TypeError:
        saved = torch.load(resume_checkpoint, map_location="cpu")

    completed_epoch_index = int(saved["epoch"])
    resume_values = json.loads(full_config.read_text())
    resume_values["epochs"] = completed_epoch_index + 2
    resume_path = Path(tempfile.gettempdir()) / "dinov3_vitb16_brats_full_resume.json"
    resume_path.write_text(json.dumps(resume_values, indent=2) + "\n")

    print("Completed epoch index:", completed_epoch_index)
    print("Next epoch index:", completed_epoch_index + 1)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "cap6411_atm.training.baseline",
            "--config",
            str(resume_path),
            "--resume",
            str(resume_checkpoint),
        ],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print("Resume skipped.")


## 7. Load a trained checkpoint and evaluate validation

This uses the checkpoint's stored validation patient IDs and the repository's global thresholded `StreamingDice`. It never constructs the test split.


In [ ]:
from dataclasses import asdict
from torch.utils.data import DataLoader
from cap6411_atm.training.metrics import StreamingDice
from cap6411_atm.training.sampling import PatientLocalBatchSampler, PatientSubset

if RUN_PRIVATE_DATA_CELLS:
    assert TRAINED_CHECKPOINT.is_file()
    try:
        trained = torch.load(TRAINED_CHECKPOINT, map_location="cpu", weights_only=False)
    except TypeError:
        trained = torch.load(TRAINED_CHECKPOINT, map_location="cpu")
    assert trained["dataset_metadata"]["test_split_loaded"] is False
    assert trained["model_config"] == asdict(model_config)

    model = build_pretrained_brats_dinov3(DINO_ROOT, PRETRAINED_CHECKPOINT, model_config)
    model.load_state_dict(trained["model_state"], strict=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()

    validation_full = BraTSAxialSliceDataset(
        BRATS_ZIP,
        SPLIT_MANIFEST,
        "validation",
        REPO_ROOT / "configs/brats2023_gli_inspection.json",
        cache_dir=None,
        max_cached_patients=1,
    )
    validation_ids = tuple(trained["dataset_metadata"]["validation_patients"])
    validation = PatientSubset(validation_full, validation_ids)
    sampler = PatientLocalBatchSampler(
        validation,
        batch_size=2,
        seed=42,
        shuffle_patients=False,
        shuffle_slices=False,
    )
    loader = DataLoader(validation, batch_sampler=sampler, num_workers=0, pin_memory=device.type == "cuda")
    metric = StreamingDice(float(trained["training_config"]["validation_threshold"]))

    with torch.inference_mode():
        for batch in loader:
            images = batch["image"].to(device, non_blocking=True)
            targets = batch["target"].to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(images)
            metric.update(logits, targets)
    print(metric.compute())
else:
    print("Validation evaluation skipped.")


## 8. Adaptive token-merging integration

Stable baseline boundary:

```python
features = model.extract_features(images)       # four × [B,768,15,15]
logits = model.decode_features(features)         # [B,3,240,240]
```

Class and four storage tokens participate inside DINOv3 but are excluded from these dense feature maps. A future merger must operate only on spatial patch tokens, preserve per-sample spatial state, and reconstruct all 225 positions in row-major 15×15 order before the current decoder. Post-encoder merging cannot accelerate the already-completed encoder. Inter-block acceleration requires a block-level wrapper and correct rotary-coordinate handling.


In [ ]:
# Weight-free identity reconstruction check for the dense decoder contract.
synthetic_features = tuple(torch.randn(2, 768, 15, 15) for _ in range(4))
reconstructed = []
for feature in synthetic_features:
    tokens = feature.flatten(2).transpose(1, 2)  # [B,225,768]
    dense = tokens.transpose(1, 2).reshape_as(feature)
    torch.testing.assert_close(dense, feature)
    reconstructed.append(dense)
print("Identity/zero-merging reconstruction preserves four dense feature maps.")


## 9. Handoff limits

- No adaptive merger or reconstructor is implemented yet.
- No encoder-acceleration claim has been made.
- The full-cohort values above are validation results selected during development, not independent test results.
- The isolated 188-patient test split remains unevaluated.
- Authorized teammates must obtain the official gated DINOv3 checkpoint and BraTS data separately.
- Do not commit `.pt`, `.pth`, `.zip`, `.nii`, `.nii.gz`, private manifests, credentials, tokens, caches, or generated outputs.
